# Notebook 1B — Swin-Tiny Severity Heatmaps (Standalone)
**Purpose:** Generate Attention-Rollout heatmaps for Swin-Tiny ONLY.
Run this notebook separately if the full Notebook 1B timed out after DenseNet / EfficientNet.

**Bug fixed:** DataLoader tuple was unpacked in wrong order in original Cell[6]:
```
# BROKEN (original):
for imgs, meta, target_classes, img_ids in loader:  # ← swapped!
    tc = int(target_classes[0])  # was actually the ISIC string → ValueError

# FIXED (this notebook):
for imgs, meta, img_ids, target_classes in loader:  # ← correct order
    tc = int(target_classes[0])  # now correctly an integer ordinal label
```

In [1]:
!pip install grad-cam -q
import os, gc, warnings
import numpy as np
import pandas as pd
from tqdm import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import timm
import cv2
import matplotlib.pyplot as plt

import albumentations as A
from albumentations.pytorch import ToTensorV2
from pytorch_grad_cam import EigenCAM
from pytorch_grad_cam.utils.image import show_cam_on_image

warnings.filterwarnings('ignore')
SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {DEVICE}")

# ── Paths — same as Notebook 1B so outputs land in the same directory ──
SEVERITY_SWIN_CKPT = '/kaggle/input/notebooks/nandini143/swin-tiny-ordinal-severity/swin_severity/swin_severity_best.pth'
SEVERITY_CSV       = '/kaggle/input/datasets/nandini143/multiclass-output/severity_data/severity_train_folds.csv'
SEVERITY_IMGDIR    = '/kaggle/input/datasets/nandini143/multiclass-output/severity_data/preprocessed_images'

OUTPUT_DIR   = '/kaggle/working/xai_outputs'
MULTI_OUT    = os.path.join(OUTPUT_DIR, 'multiclass')
SWIN_OUT_DIR = os.path.join(MULTI_OUT, 'heatmaps_swin')
os.makedirs(SWIN_OUT_DIR, exist_ok=True)

IMAGENET_MEAN      = [0.485, 0.456, 0.406]
IMAGENET_STD       = [0.229, 0.224, 0.225]
IMG_SIZE_SWIN      = 224
SEVERITY_NUM_OUTPUTS = 3

print("Configuration loaded.")
print(f"  Swin checkpoint : {SEVERITY_SWIN_CKPT}")
print(f"  Output dir      : {SWIN_OUT_DIR}")


Device: cuda
Configuration loaded.
  Swin checkpoint : /kaggle/input/notebooks/nandini143/swin-tiny-ordinal-severity/swin_severity/swin_severity_best.pth
  Output dir      : /kaggle/working/xai_outputs/multiclass/heatmaps_swin


In [2]:
print("Loading metadata...")
sev_df   = pd.read_csv(SEVERITY_CSV)
site_cols = [c for c in sev_df.columns if c.startswith('site_')]
actual_meta_dim = 3 + len(site_cols)
print(f"  Rows: {len(sev_df)}  |  meta_dim: {actual_meta_dim}  |  site_cols: {len(site_cols)}")

sev_df['img_path_resolved'] = sev_df['image'].apply(
    lambda iid: os.path.join(SEVERITY_IMGDIR, f'{iid}.jpg')
)

class SeverityXAIDataset(Dataset):
    """Identical to Notebook 1B Cell[1]. Returns (img, meta, image_id, target_class)."""
    def __init__(self, df, transform, site_cols):
        self.df        = df.reset_index(drop=True)
        self.transform = transform
        self.site_cols = site_cols

    def __len__(self): return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img = cv2.imread(str(row['img_path_resolved']))
        if img is None:
            img = np.zeros((IMG_SIZE_SWIN, IMG_SIZE_SWIN, 3), dtype=np.uint8)
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        if self.transform:
            img = self.transform(image=img)['image']

        meta_vals  = [float(row.get('age_norm', 0.5)),
                      float(row.get('sex_male',  0.0)),
                      float(row.get('sex_female', 0.0))]
        meta_vals += [float(row.get(c, 0.0)) for c in self.site_cols]
        meta       = torch.tensor(meta_vals, dtype=torch.float32)

        # ordinal_label is the integer class index (0,1,2,3)
        target_cls = int(row.get('ordinal_label', 3))

        # Return order: img, meta, image_id (string), target_cls (int)
        return img, meta, row['image'], target_cls

sev_transform_swin = A.Compose([
    A.Resize(IMG_SIZE_SWIN, IMG_SIZE_SWIN),
    A.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
    ToTensorV2(),
])

print("Dataset class and transform ready.")


Loading metadata...
  Rows: 9340  |  meta_dim: 12  |  site_cols: 9
Dataset class and transform ready.


In [3]:
class TabularMLP(nn.Module):
    def __init__(self, in_dim, hidden_dim=64, out_dim=32, dropout=0.2, use_gelu=True):
        super().__init__()
        act = nn.GELU() if use_gelu else nn.ReLU(inplace=True)
        self.net = nn.Sequential(
            nn.Linear(in_dim, hidden_dim), nn.BatchNorm1d(hidden_dim), act,
            nn.Dropout(dropout), nn.Linear(hidden_dim, out_dim), act,
        )
    def forward(self, x): return self.net(x)


class SwinSeverityModel(nn.Module):
    """Exact replica of the Swin model used in Notebook 1B training."""
    def __init__(self, model_name, num_outputs, meta_dim,
                 pretrained=False, drop_rate=0.2, drop_path_rate=0.2):
        super().__init__()
        self.backbone = timm.create_model(
            model_name, pretrained=pretrained, num_classes=0,
            drop_rate=drop_rate, drop_path_rate=drop_path_rate
        )
        feat_dim = self.backbone.num_features
        self.use_meta = meta_dim > 0
        if self.use_meta:
            self.meta_mlp  = TabularMLP(meta_dim, 64, 32, 0.2, use_gelu=True)
            fusion_dim     = feat_dim + 32
        else:
            fusion_dim = feat_dim
        self.head = nn.Sequential(
            nn.LayerNorm(fusion_dim), nn.Linear(fusion_dim, 512), nn.GELU(),
            nn.Dropout(0.3), nn.Linear(512, 128), nn.GELU(),
            nn.Dropout(0.1), nn.Linear(128, num_outputs),
        )

    def forward(self, images, meta=None):
        feat = self.backbone(images)
        if self.use_meta and meta is not None:
            feat = torch.cat([feat, self.meta_mlp(meta)], dim=1)
        return self.head(feat)


def load_swin_safely(ckpt_path, meta_dim, device):
    model = SwinSeverityModel('swin_tiny_patch4_window7_224', SEVERITY_NUM_OUTPUTS, meta_dim)
    ckpt  = torch.load(ckpt_path, map_location=device, weights_only=False)
    state = ckpt.get('model_state', ckpt.get('model_state_dict', ckpt))

    # Shape-safe loading: skip layers with dimension mismatches
    model_dict     = model.state_dict()
    filtered_state = {
        k: v for k, v in state.items()
        if k in model_dict and v.shape == model_dict[k].shape
    }
    skipped = [k for k in state if k not in filtered_state and k in model_dict]
    if skipped:
        print(f"  Skipped {len(skipped)} mismatched keys: {skipped}")

    model_dict.update(filtered_state)
    model.load_state_dict(model_dict, strict=False)
    model.to(device).eval()
    print(f"Swin-Tiny loaded. Matched {len(filtered_state)}/{len(state)} keys.")
    return model

print("Model definitions ready.")


Model definitions ready.


In [ ]:
class SwinWrapper(nn.Module):
    def __init__(self, model, meta):
        super().__init__()
        self.model = model
        self.meta = meta
        
    def forward(self, x):
        return self.model(x, meta=self.meta)

def reshape_transform(tensor):
    if tensor.ndim == 4:
        return tensor.permute(0, 3, 1, 2)
    elif tensor.ndim == 3:
        B, L, C = tensor.shape
        H = int(np.sqrt(L))
        return tensor.reshape(B, H, H, C).permute(0, 3, 1, 2)
    return tensor

print("SwinWrapper and reshape_transform ready.")


In [ ]:
print("[1/1] Loading Swin-Tiny model...")
sev_swin_model = load_swin_safely(SEVERITY_SWIN_CKPT, actual_meta_dim, DEVICE)

target_layers = [sev_swin_model.backbone.layers[-1].blocks[-1]]

# ── Resume Support: skip already-processed files ──
already_done = {f.replace('.npy', '') for f in os.listdir(SWIN_OUT_DIR) if f.endswith('.npy')}
print(f"  Already processed : {len(already_done)} files (will skip these)")

sev_loader_swin = DataLoader(
    SeverityXAIDataset(sev_df, sev_transform_swin, site_cols),
    batch_size=1, shuffle=False, num_workers=2, pin_memory=True
)

skipped = 0
processed = 0
errors    = 0

for imgs, meta, img_ids, target_classes in tqdm(sev_loader_swin, desc='Sev-Swin EigenCAM'):
    image_id = img_ids[0]
    tc       = int(target_classes[0])
    
    if image_id in already_done:
        skipped += 1
        continue
    
    try:
        imgs = imgs.to(DEVICE)
        meta = meta.to(DEVICE)
        
        wrapped_model = SwinWrapper(sev_swin_model, meta)
        cam = EigenCAM(model=wrapped_model, target_layers=target_layers, reshape_transform=reshape_transform)
        
        grayscale_cam = cam(input_tensor=imgs)[0, :]
        
        os.makedirs(SWIN_OUT_DIR, exist_ok=True)
        np.save(os.path.join(SWIN_OUT_DIR, f'{image_id}.npy'), grayscale_cam)
        
        processed += 1
    except Exception as e:
        print(f"  [WARN] {image_id} failed: {e}")
        errors += 1
        continue

del sev_swin_model
gc.collect()
torch.cuda.empty_cache()

print(f"\nSwin Severity Heatmaps complete!")
print(f"  Processed : {processed}")
print(f"  Skipped   : {skipped}  (already existed)")
print(f"  Errors    : {errors}")
print(f"  Total .npy: {len([f for f in os.listdir(SWIN_OUT_DIR) if f.endswith('.npy')])}")


In [ ]:
print("All images processed!")


In [7]:
import shutil

count = len([f for f in os.listdir(SWIN_OUT_DIR) if f.endswith('.npy')])
print(f"Total Swin heatmaps saved: {count}")
shutil.make_archive('/kaggle/working/swin_severity_heatmaps', 'zip', SWIN_OUT_DIR)
print("Archive saved: swin_severity_heatmaps.zip")


Total Swin heatmaps saved: 9340
Archive saved: swin_severity_heatmaps.zip
